# 01 — Data Prep (D1–D2)

**VinDr-CXR → YOLO format.** Positive-only subset, WBF rater fusion, stratified 80/10/10.

> ⚠️ **This notebook is where the paper quietly dies.** If rater fusion is wrong, every downstream number is wrong, and the D4 sanity gate will *not* catch it — broken-but-plausible labels give a broken-but-plausible 0.25 mAP that looks fine.

**Settings:** accelerator `None` (CPU only), internet ON.

**Gate:** §8 `verify()` must pass before notebook 02.

---
### Changed since the first run (2026-07-20)

Visual inspection of 12 fused images showed **systematic under-merging at IoU 0.5** — same-class rater boxes covering one finding surviving as 2–4 separate ground-truth objects (Pulmonary fibrosis, Pleural thickening, Pleural effusion, Nodule/Mass). One measured pair had IoU 0.38, just under the cut.

- `FUSION_IOU` **0.5 → 0.4** in `config.py`
- new §5 sweep + duplicate audit — confirm the threshold rather than trusting it
- new §6 targeted re-plot of the *worst* offenders, not random samples
- paths moved into `config.py` (were overridden in-notebook, which is the drift pattern)

In [ ]:
!pip install -q ensemble-boxes

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np

from src import config as C
from src.data import fusion, prepare

!cd {REPO} && git rev-parse --short HEAD    # commit hash -> paper
print(C.as_dict())

## 1. Load raw annotations

Paths live in `config.py` now. If they need changing, change them **there** and re-push — not here. The moment values get patched in the notebook, the frozen-protocol guarantee is gone.

> `train.csv` carries two coordinate systems. `x_min/y_min/x_max/y_max` are already scaled to the 1024px JPGs — use those. The `raw_*` columns are original DICOM space; using them against the resized images would silently give boxes 2–3× too large.

In [ ]:
IMAGE_DIR = C.IMAGE_DIR
assert C.TRAIN_CSV.exists(), f"not found: {C.TRAIN_CSV} -- fix config.py"

df_raw = prepare.load_raw(C.TRAIN_CSV)
df_raw.head()

## 2. Rater distribution

**Verified 2026-07-20:** R9/R10/R8 produced **34,462 of 36,096 positive boxes = 95.5%**. Only 11 raters appear at all; R1–R7 are effectively absent (R2 contributes 3 boxes).

This is stronger than `dataset-research-notes.md` originally claimed. **Belongs in the paper's limitations** — the fused labels are effectively three radiologists' opinions, not a 17-rater consensus.

In [ ]:
if "rad_id" in df_raw.columns:
    vc = df_raw[df_raw.class_id != 14].rad_id.value_counts()
    print(vc)
    top3 = vc.head(3).sum()
    print(f"\ntop-3 raters: {top3}/{vc.sum()} = {top3/vc.sum():.1%} of positive boxes")

## 3. Positive-only subset

Expect **4,394 images (29.3%)**. The compute enabler — ~3.4× epoch-time reduction, and the established protocol here (`sunghyunjun` trained its detector on abnormal images only).

**Report this.** The model never sees normal images and cannot be evaluated on normal-vs-abnormal discrimination.

In [ ]:
df_pos = prepare.positive_only(df_raw)
dims = prepare.get_image_dims(IMAGE_DIR, df_pos.image_id.unique())
print(f"{len(dims)} images measured")

## 4. Fusion threshold sweep — NEW

Don't trust the threshold, measure it.

**Reading the output — look for the knee:**
- `dup_pairs` should fall sharply as the threshold drops
- `n_boxes` must **not** collapse toward the printed floor (~12,000) — that would mean every 3-rater group merges unconditionally, including genuinely distinct lesions

Pick the lowest threshold that kills duplicates while retention stays comfortably above ~40%.

In [ ]:
sweep = fusion.sweep_fusion_iou(df_pos, dims, thresholds=(0.25, 0.3, 0.4, 0.5))
sweep

> **If the sweep disagrees with `FUSION_IOU = 0.4`**, change it in `config.py`, add a CHANGELOG line, push, and re-run this notebook from the top. Do not override it in the next cell.

## 5. Fuse at the chosen threshold

**Unsettled, state it in the paper:** the `sunghyunjun` reference's own ablation has WBF *winning* on private LB (0.185 vs 0.181) while *losing* on CV (0.4158 vs 0.4419). The author chose NMS by reasoning about test-set labeling, not measurement. Don't inherit either as established.

In [ ]:
df_fused = fusion.fuse_dataset(df_pos, dims, method=C.FUSION_METHOD, iou_thr=C.FUSION_IOU)

audit = fusion.count_near_duplicates(df_fused, iou_thr=C.DUPLICATE_AUDIT_IOU)
print(f"\npost-fusion audit: {audit['dup_pairs']} near-duplicate pairs "
      f"across {audit['dup_images']} images")
print("worst classes:", {C.CLASSES[k]: v for k, v in list(audit['dup_per_class'].items())[:5]})

## 6. Fusion report — sanity check #1

**Baseline from the IoU 0.5 run**, for comparison:

| | retention |
|---|---|
| Cardiomegaly, Aortic enlargement | 44–47% (merging well) |
| Pleural thickening, Other lesion, Atelectasis, Lung Opacity | 81–84% (barely merging) |
| **overall** | **66.3%** |

At 0.4 the diffuse classes should drop toward the anchored ones. Large findings merge because raters agree where the heart is; diffuse findings didn't because raters disagree on extent.

In [ ]:
rep = fusion.fusion_report(df_pos, df_fused)
rep

## 7. Visual check — sanity check #2 (MANDATORY)

Now targeted at the **worst offenders**, not random samples. Random sampling is why the first pass nearly missed this.

Red dotted = individual rater boxes. Green = fused.

**Looking for:** duplicate near-identical green boxes of the same class stacked in one region. That was the failure at 0.5 — `Pulmonary fibrosis` ×2, `Pleural effusion` ×2, `Pleural thickening`/`Nodule/Mass` stacked 3–4 deep.

**Not a problem:** same-class boxes in *different* anatomical regions (ILD in both lungs). Those are genuinely distinct findings, IoU ≈ 0, and no threshold should merge them.

In [ ]:
worst = fusion.find_duplicate_images(df_fused, iou_thr=C.DUPLICATE_AUDIT_IOU, limit=6)
print(f"{len(worst)} images still showing duplicates" if worst else "no duplicates found ✓")
if worst:
    fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, worst)

In [ ]:
# Random sample too -- confirms fusion didn't over-merge elsewhere.
rng = np.random.default_rng(C.SEED)
sample_ids = list(rng.choice(df_fused.image_id.unique(), size=6, replace=False))
fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, sample_ids)

## 8. Stratified split + YOLO export

Stratified on each image's **rarest** present class — protects tail classes from vanishing out of val/test, which would make their per-class AP undefined and break the Axis-A analysis.

> ⚠️ At IoU 0.5 the test split held only **12 Pneumothorax** and **24 Atelectasis** boxes. Per-class AP on counts that small is noise. Report those with explicit *n*, or aggregate them.

In [ ]:
splits = prepare.stratified_split(df_fused, split=C.SPLIT, seed=C.SEED)
dist = prepare.class_distribution(df_fused, splits)
dist

In [ ]:
thin = dist[dist.test < 30][["class", "train", "val", "test"]]
if len(thin):
    print("⚠ classes with <30 test boxes -- per-class AP will be unstable:")
    print(thin)

In [ ]:
C.DATA_ROOT.mkdir(parents=True, exist_ok=True)
prepare.to_yolo_labels(df_fused, dims, splits, C.DATA_ROOT, IMAGE_DIR)
data_yaml = prepare.write_data_yaml(C.DATA_ROOT, C.CLASSES)

## 9. Verify — GATE

Must print `✓ verify passed` before notebook 02.

In [ ]:
ok = prepare.verify(C.DATA_ROOT, splits)
assert ok, "data prep failed verification -- do NOT proceed to training"

df_fused.to_csv(C.DATA_ROOT / "fused_boxes.csv", index=False)
pd.Series(splits).to_json(C.DATA_ROOT / "splits.json")

# Provenance -- these numbers go in the paper's dataset section.
import json
(C.DATA_ROOT / "prep_manifest.json").write_text(json.dumps({
    **C.as_dict(),
    "n_raw_rows": len(df_raw),
    "n_positive_images": len(dims),
    "n_raw_boxes": int((df_pos.class_id != 14).sum()),
    "n_fused_boxes": len(df_fused),
    "duplicate_audit": audit,
    "split_sizes": {k: len(v) for k, v in splits.items()},
}, indent=2, default=str))

print(f"\ndone. data.yaml -> {data_yaml}")

## 10. Save as Kaggle Dataset

`File → Save Version → Save Output`, then attach the output as an input dataset in notebook 02.

`/kaggle/working` dies with the 12h session, and rebuilding costs an hour you won't have on D5.

**Keep it private** — VinDr-CXR carries a data use agreement restricting redistribution.